# Notebook 09 — Fatigue Progression & Temporal Analysis



## 0. Imports and configuration

In [1]:
from pathlib import Path
from collections import deque
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FUSION_CONFIG_PATH = (
    OUTPUT_DIR / "final_fusion_config.json"
)

print("Project root:", PROJECT_ROOT)
print("Fusion config:", FUSION_CONFIG_PATH)


Project root: d:\driver-drowsiness-detection
Fusion config: d:\driver-drowsiness-detection\outputs\final_fusion_config.json


## 1. Load the final fusion configuration

In [2]:
if not FUSION_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Run Notebook 08 first. "
        f"Missing: {FUSION_CONFIG_PATH}"
    )

fusion_config = json.loads(
    FUSION_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        fusion_config,
        indent=2
    )
)


{
  "eye_model": {
    "name": "Eye MobileNetV2",
    "path": "d:\\driver-drowsiness-detection\\models\\eye_mobilenetv2_best.keras",
    "classes": [
      "Closed",
      "Open"
    ]
  },
  "mouth_model": {
    "name": "Mouth MobileNetV2",
    "path": "d:\\driver-drowsiness-detection\\models\\mouth_mobilenetv2_best.keras",
    "classes": [
      "no_yawn",
      "yawn"
    ]
  },
  "confidence_threshold": 0.75,
  "outputs": [
    "Alert",
    "Mild Fatigue",
    "Severe Fatigue",
    "Uncertain"
  ],
  "decision_rules": {
    "closed_eye": "Severe Fatigue",
    "open_eye_yawn": "Mild Fatigue",
    "open_eye_no_yawn": "Alert",
    "insufficient_or_conflicting_confidence": "Uncertain"
  },
  "end_to_end_fusion_accuracy": null,
  "end_to_end_fusion_note": "Requires paired eye and mouth ground truth from the same frames."
}


## 2. Temporal-analysis configuration




In [3]:
TEMPORAL_CONFIG = {
    "history_size": 15,

    # Number of recent observations needed
    # before a persistent closed-eye condition
    # is treated as severe fatigue.
    "severe_closed_count": 3,

    # Number of recent observations needed
    # before repeated yawning is treated as mild fatigue.
    "mild_yawn_count": 3,

    # Fraction of recent valid observations that
    # must support the condition.
    "severe_ratio": 0.50,
    "mild_ratio": 0.40,

    # How many recent observations are inspected
    # for escalation.
    "severe_window": 5,
    "mild_window": 7
}

print(
    json.dumps(
        TEMPORAL_CONFIG,
        indent=2
    )
)


{
  "history_size": 15,
  "severe_closed_count": 3,
  "mild_yawn_count": 3,
  "severe_ratio": 0.5,
  "mild_ratio": 0.4,
  "severe_window": 5,
  "mild_window": 7
}


## 4. Temporal fatigue analyzer


### Decision order

```text
1. Repeated severe closed-eye evidence
             ↓
       Severe Fatigue

2. Repeated yawn evidence
             ↓
        Mild Fatigue

3. Otherwise recent evidence is stable
             ↓
           Alert

4. Insufficient / ambiguous evidence
             ↓
         Uncertain
```

A single isolated yawn or closed-eye observation should not automatically escalate the state.


In [5]:
class TemporalFatigueAnalyzer:

    def __init__(self, config):
        self.config = config

        self.history = deque(
            maxlen=config["history_size"]
        )

    def reset(self):
        self.history.clear()

    def add_observation(self, observation):
        self.history.append(observation)
        return self.current_state()

    def _recent(self, count):
        history = list(self.history)
        return history[-count:]

    def current_state(self):

        if not self.history:
            return {
                "fatigue_state": "Uncertain",
                "reason": "No observations available",
                "history_length": 0,
                "closed_count": 0,
                "yawn_count": 0
            }

        severe_window = self._recent(
            self.config["severe_window"]
        )

        mild_window = self._recent(
            self.config["mild_window"]
        )

        # Count confident closed-eye evidence.
        closed_count = sum(
            1
            for item in severe_window
            if (
                item["eye_state"] == "Closed"
                and item["eye_confidence"]
                >= 0.75
            )
        )

        # Count confident yawn evidence.
        yawn_count = sum(
            1
            for item in mild_window
            if (
                item["mouth_state"] == "yawn"
                and item["mouth_confidence"]
                >= 0.75
            )
        )

        severe_ratio = (
            closed_count / len(severe_window)
            if severe_window
            else 0.0
        )

        mild_ratio = (
            yawn_count / len(mild_window)
            if mild_window
            else 0.0
        )

        if (
            closed_count
            >= self.config["severe_closed_count"]
            and severe_ratio
            >= self.config["severe_ratio"]
        ):
            state = "Severe Fatigue"
            reason = (
                "Repeated confident closed-eye "
                "observations detected."
            )

        elif (
            yawn_count
            >= self.config["mild_yawn_count"]
            and mild_ratio
            >= self.config["mild_ratio"]
        ):
            state = "Mild Fatigue"
            reason = (
                "Repeated confident yawn "
                "observations detected."
            )

        else:
            recent_states = [
                item["fatigue_state"]
                for item in self._recent(3)
            ]

            if (
                recent_states
                and all(
                    state == "Alert"
                    for state in recent_states
                )
            ):
                state = "Alert"
                reason = (
                    "Recent observations consistently "
                    "indicate alert state."
                )
            else:
                state = "Uncertain"
                reason = (
                    "Evidence is not persistent enough "
                    "for a temporal fatigue escalation."
                )

        return {
            "fatigue_state": state,
            "reason": reason,
            "history_length": len(self.history),
            "closed_count": closed_count,
            "yawn_count": yawn_count,
            "closed_ratio": round(
                severe_ratio,
                4
            ),
            "yawn_ratio": round(
                mild_ratio,
                4
            )
        }


print("✓ Temporal fatigue analyzer created.")


✓ Temporal fatigue analyzer created.


## 5. Test: stable alert sequence



In [6]:
def make_observation_sequence(
    count,
    fatigue_state,
    eye_state,
    mouth_state,
    eye_confidence=0.95,
    mouth_confidence=0.95,
    start_frame=1
):
    observations = []

    for i in range(count):
        observations.append(
            normalize_observation(
                frame_number=start_frame + i,
                fatigue_state=fatigue_state,
                eye_state=eye_state,
                mouth_state=mouth_state,
                eye_confidence=eye_confidence,
                mouth_confidence=mouth_confidence
            )
        )

    return observations


alert_sequence = make_observation_sequence(
    count=10,
    fatigue_state="Alert",
    eye_state="Open",
    mouth_state="no_yawn"
)

analyzer = TemporalFatigueAnalyzer(
    TEMPORAL_CONFIG
)

alert_results = []

for observation in alert_sequence:
    result = analyzer.add_observation(
        observation
    )
    alert_results.append(result["fatigue_state"])

print(alert_results)
print(
    "Final state:",
    alert_results[-1]
)


['Alert', 'Alert', 'Alert', 'Alert', 'Alert', 'Alert', 'Alert', 'Alert', 'Alert', 'Alert']
Final state: Alert


## 6. Test: isolated yawn should not immediately escalate



In [7]:
sequence = []

sequence.extend(
    make_observation_sequence(
        count=5,
        fatigue_state="Alert",
        eye_state="Open",
        mouth_state="no_yawn",
        start_frame=1
    )
)

sequence.extend(
    make_observation_sequence(
        count=1,
        fatigue_state="Mild Fatigue",
        eye_state="Open",
        mouth_state="yawn",
        start_frame=6
    )
)

sequence.extend(
    make_observation_sequence(
        count=5,
        fatigue_state="Alert",
        eye_state="Open",
        mouth_state="no_yawn",
        start_frame=7
    )
)

analyzer.reset()

results = []

for observation in sequence:
    result = analyzer.add_observation(
        observation
    )

    results.append({
        "frame": observation["frame"],
        "frame_state": observation["fatigue_state"],
        "temporal_state": result["fatigue_state"],
        "yawn_count": result["yawn_count"]
    })

isolated_yawn_df = pd.DataFrame(
    results
)

display(isolated_yawn_df)

print(
    "Final temporal state:",
    results[-1]["temporal_state"]
)


,frame,frame_state,temporal_state,yawn_count
0,1,Alert,Alert,0
1,2,Alert,Alert,0
2,3,Alert,Alert,0
3,4,Alert,Alert,0
4,5,Alert,Alert,0
5,6,Mild Fatigue,Uncertain,1
6,7,Alert,Uncertain,1
7,8,Alert,Uncertain,1
8,9,Alert,Alert,1
9,10,Alert,Alert,1


Final temporal state: Alert


## 7. Test: persistent yawning should progress to Mild Fatigue



In [8]:
sequence = []

sequence.extend(
    make_observation_sequence(
        count=4,
        fatigue_state="Alert",
        eye_state="Open",
        mouth_state="no_yawn",
        start_frame=1
    )
)

sequence.extend(
    make_observation_sequence(
        count=5,
        fatigue_state="Mild Fatigue",
        eye_state="Open",
        mouth_state="yawn",
        start_frame=5
    )
)

analyzer.reset()

results = []

for observation in sequence:
    result = analyzer.add_observation(
        observation
    )

    results.append({
        "frame": observation["frame"],
        "frame_state": observation["fatigue_state"],
        "temporal_state": result["fatigue_state"],
        "yawn_count": result["yawn_count"],
        "yawn_ratio": result["yawn_ratio"]
    })

persistent_yawn_df = pd.DataFrame(
    results
)

display(persistent_yawn_df)

print(
    "Final temporal state:",
    results[-1]["temporal_state"]
)


,frame,frame_state,temporal_state,yawn_count,yawn_ratio
0,1,Alert,Alert,0,0.0000
1,2,Alert,Alert,0,0.0000
2,3,Alert,Alert,0,0.0000
3,4,Alert,Alert,0,0.0000
4,5,Mild Fatigue,Uncertain,1,0.2000
5,6,Mild Fatigue,Uncertain,2,0.3333
6,7,Mild Fatigue,Mild Fatigue,3,0.4286
7,8,Mild Fatigue,Mild Fatigue,4,0.5714
8,9,Mild Fatigue,Mild Fatigue,5,0.7143


Final temporal state: Mild Fatigue


## 8. Test: persistent closed eyes should progress to Severe Fatigue


In [9]:
sequence = []

sequence.extend(
    make_observation_sequence(
        count=4,
        fatigue_state="Alert",
        eye_state="Open",
        mouth_state="no_yawn",
        start_frame=1
    )
)

sequence.extend(
    make_observation_sequence(
        count=6,
        fatigue_state="Severe Fatigue",
        eye_state="Closed",
        mouth_state="no_yawn",
        start_frame=5
    )
)

analyzer.reset()

results = []

for observation in sequence:
    result = analyzer.add_observation(
        observation
    )

    results.append({
        "frame": observation["frame"],
        "frame_state": observation["fatigue_state"],
        "temporal_state": result["fatigue_state"],
        "closed_count": result["closed_count"],
        "closed_ratio": result["closed_ratio"]
    })

persistent_closed_df = pd.DataFrame(
    results
)

display(persistent_closed_df)

print(
    "Final temporal state:",
    results[-1]["temporal_state"]
)


,frame,frame_state,temporal_state,closed_count,closed_ratio
0,1,Alert,Alert,0,0.0
1,2,Alert,Alert,0,0.0
2,3,Alert,Alert,0,0.0
3,4,Alert,Alert,0,0.0
4,5,Severe Fatigue,Uncertain,1,0.2
5,6,Severe Fatigue,Uncertain,2,0.4
6,7,Severe Fatigue,Severe Fatigue,3,0.6
7,8,Severe Fatigue,Severe Fatigue,4,0.8
8,9,Severe Fatigue,Severe Fatigue,5,1.0
9,10,Severe Fatigue,Severe Fatigue,5,1.0


Final temporal state: Severe Fatigue


## 9. Test: closed eyes + yawn


In [10]:
sequence = []

for frame in range(1, 8):
    sequence.append(
        normalize_observation(
            frame_number=frame,
            fatigue_state="Severe Fatigue",
            eye_state="Closed",
            mouth_state="yawn",
            eye_confidence=0.97,
            mouth_confidence=0.96
        )
    )

analyzer.reset()

results = []

for observation in sequence:
    result = analyzer.add_observation(
        observation
    )

    results.append({
        "frame": observation["frame"],
        "temporal_state": result["fatigue_state"],
        "reason": result["reason"]
    })

display(
    pd.DataFrame(results)
)

print(
    "Final state:",
    results[-1]["temporal_state"]
)


,frame,temporal_state,reason
0,1,Uncertain,Evidence is not persistent enough for a tempor...
1,2,Uncertain,Evidence is not persistent enough for a tempor...
2,3,Severe Fatigue,Repeated confident closed-eye observations det...
3,4,Severe Fatigue,Repeated confident closed-eye observations det...
4,5,Severe Fatigue,Repeated confident closed-eye observations det...
5,6,Severe Fatigue,Repeated confident closed-eye observations det...
6,7,Severe Fatigue,Repeated confident closed-eye observations det...


Final state: Severe Fatigue


## 10. Test: uncertain observations


In [11]:
uncertain_sequence = []

for frame in range(1, 8):
    uncertain_sequence.append(
        normalize_observation(
            frame_number=frame,
            fatigue_state="Uncertain",
            eye_state="Open",
            mouth_state="no_yawn",
            eye_confidence=0.58,
            mouth_confidence=0.61
        )
    )

analyzer.reset()

results = []

for observation in uncertain_sequence:
    result = analyzer.add_observation(
        observation
    )

    results.append({
        "frame": observation["frame"],
        "temporal_state": result["fatigue_state"],
        "reason": result["reason"]
    })

display(
    pd.DataFrame(results)
)

print(
    "Final state:",
    results[-1]["temporal_state"]
)


,frame,temporal_state,reason
0,1,Uncertain,Evidence is not persistent enough for a tempor...
1,2,Uncertain,Evidence is not persistent enough for a tempor...
2,3,Uncertain,Evidence is not persistent enough for a tempor...
3,4,Uncertain,Evidence is not persistent enough for a tempor...
4,5,Uncertain,Evidence is not persistent enough for a tempor...
5,6,Uncertain,Evidence is not persistent enough for a tempor...
6,7,Uncertain,Evidence is not persistent enough for a tempor...


Final state: Uncertain


## 12. Create a reusable temporal-processing function



In [14]:
def process_frame_result(
    analyzer,
    frame_result
):
    return analyzer.add_observation(
        frame_result
    )


# Example
demo_analyzer = TemporalFatigueAnalyzer(
    TEMPORAL_CONFIG
)

demo_frame = normalize_observation(
    frame_number=100,
    fatigue_state="Mild Fatigue",
    eye_state="Open",
    mouth_state="yawn",
    eye_confidence=0.93,
    mouth_confidence=0.91
)

demo_output = process_frame_result(
    demo_analyzer,
    demo_frame
)

print(
    json.dumps(
        demo_output,
        indent=2
    )
)


{
  "fatigue_state": "Uncertain",
  "reason": "Evidence is not persistent enough for a temporal fatigue escalation.",
  "history_length": 1,
  "closed_count": 0,
  "yawn_count": 1,
  "closed_ratio": 0.0,
  "yawn_ratio": 1.0
}


## 13. Save temporal configuration


In [15]:
temporal_config_path = (
    OUTPUT_DIR /
    "temporal_fatigue_config.json"
)

temporal_config_path.write_text(
    json.dumps(
        TEMPORAL_CONFIG,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Saved:",
    temporal_config_path
)


Saved: d:\driver-drowsiness-detection\outputs\temporal_fatigue_config.json


## 14. Save temporal-analysis report

In [16]:
temporal_report = {
    "temporal_accuracy": None,
    "evaluation_status": (
        "Logic validated with controlled sequences; "
        "no paired labeled video dataset is available "
        "for measured temporal accuracy."
    ),
    "tests": {
        "stable_alert": True,
        "isolated_yawn": True,
        "persistent_yawn": True,
        "persistent_closed_eyes": True,
        "closed_eye_plus_yawn": True,
        "uncertain_observations": True
    },
    "configuration": TEMPORAL_CONFIG,
    "progression_demo": [
        "Alert",
        "Mild Fatigue",
        "Severe Fatigue"
    ]
}

report_path = (
    OUTPUT_DIR /
    "temporal_analysis_report.json"
)

report_path.write_text(
    json.dumps(
        temporal_report,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        temporal_report,
        indent=2
    )
)

print(
    "\nSaved:",
    report_path
)


{
  "temporal_accuracy": null,
  "evaluation_status": "Logic validated with controlled sequences; no paired labeled video dataset is available for measured temporal accuracy.",
  "tests": {
    "stable_alert": true,
    "isolated_yawn": true,
    "persistent_yawn": true,
    "persistent_closed_eyes": true,
    "closed_eye_plus_yawn": true,
    "uncertain_observations": true
  },
  "configuration": {
    "history_size": 15,
    "severe_closed_count": 3,
    "mild_yawn_count": 3,
    "severe_ratio": 0.5,
    "mild_ratio": 0.4,
    "severe_window": 5,
    "mild_window": 7
  },
  "progression_demo": [
    "Alert",
    "Mild Fatigue",
    "Severe Fatigue"
  ]
}

Saved: d:\driver-drowsiness-detection\outputs\temporal_analysis_report.json
